In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = "finance-bok-test"
NAMESPACE  = "bok-ns1"   

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

# 기존 인덱스 없으면 생성 (있으면 재사용)
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"{INDEX_NAME} 생성 완료")
else:
    print(f"{INDEX_NAME} 이미 존재함 — 재사용")

finance-bok-test 생성 완료


In [3]:
# 상태 확인
f_index = pc.Index(INDEX_NAME)
print(f_index.describe_index_stats())

c:\Users\Admin\miniconda3\envs\langchain_rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [6]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print(f"총 페이지 수: {len(docs)}")
print(f"\n첫 페이지 내용 (앞 200자):\n{docs[0].page_content[:200]}")

총 페이지 수: 93

첫 페이지 내용 (앞 200자):
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

splits = splitter.split_documents(docs)
print(f"총 청크 수: {len(splits)}")

총 청크 수: 249


In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 중복 업서트 방지: namespace에 데이터가 없을 때만 업서트
stats = f_index.describe_index_stats()
vector_count = stats['namespaces'].get(NAMESPACE, {}).get('vector_count', 0)

if vector_count == 0:
    print("업서트 시작...")
    vectorstore = PineconeVectorStore.from_documents(
        documents=splits,
        embedding=embedding_model,
        index_name=INDEX_NAME,
        namespace=NAMESPACE
    )
    print(f"업서트 완료 — 총 {len(splits)}개 청크")
else:
    print(f"이미 {vector_count}개 벡터 존재 — 업서트 건너뜀")
    vectorstore = PineconeVectorStore.from_existing_index(
        index_name=INDEX_NAME,
        embedding=embedding_model,
        namespace=NAMESPACE
    )

업서트 시작...
업서트 완료 — 총 249개 청크


In [9]:
# 업서트 후 상태 확인
print(f_index.describe_index_stats())

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'bok-ns1': {'vector_count': 249}},
 'total_vector_count': 249,
 'vector_type': 'dense'}


In [10]:
# Retriever 설정 (k=5로 개선)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5, "namespace": NAMESPACE}
)

In [11]:
# 검색 테스트
queries = [
    "2026년 경제성장률 전망은?",
    "물가 상승률 전망은?",
    "미국 관세가 한국 경제에 미치는 영향은?"
]

for query in queries:
    print(f"[ 질문 ] {query}")
    results = retriever.invoke(query)
    for i, r in enumerate(results):
        print(f"  {i+1}. (p.{r.metadata.get('page', '')}) {r.page_content[:100]}")
    print()

[ 질문 ] 2026년 경제성장률 전망은?
  1. (p.43.0) 30 
 
<경제성장 전망1)> 
(전년동기대비, %) 
  2024 2025 2026e) 2027e) 
연간 상반 하반 연간 상반 하반 연간 연간 
GDP 성장률 2.0 0.3 
  2. (p.35.0) 22 
 
2. 거시경제 전망 
 
 
 
 
경제성장 
 
2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한
  3. (p.6.0) < 요약 1/8 > 
 
  
경제전망 요약 
 올해 우리 경제는 美관세 영향과 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한 세계경제 흐름 등에
  4. (p.13.0) < 요약 8/8 > 
  
 
 2024  2025  2026e)  2027e) 
연간  상반 하반1) 연간1)2)  상반 하반 연간2)  연간2) 
              <전
  5. (p.9.0) ▪ 27년에는 내수 회복세가 지속되는 가운데 수출도 세계경제 성장세 지속, 반도체 
공급능력 확충 등으로 증가하며 1.8%의 견조한 성장세를 나타낼 전망이다. 
 
 
 
<국내 

[ 질문 ] 물가 상승률 전망은?
  1. (p.53.0) 1.9% 대비 0.1%p 높아졌다. 소비자물가 상승률의 경우에도 11월 전망1.9% 대비 0.1%p 높
아진 2.0%로 나타났다. 
 
시장의 국내 성장 및 물가 전망 모두 올해 
  2. (p.53.0) 40 
 
3. 전망의 리스크 평가 
    
주요 리스크 요인 
 
3-1. 향후 성장 전망경로에는 반도체 경기, 글로벌 통상환경, 국제금융시장 등과 관련
한 불확실성이 크며, 
  3. (p.11.0) < 요약 6/8 > 
6  
  
전망의 리스크 
 
 향후 성장 전망경로에는 반도체 경기, 글로벌 통상환경, 국제금융시장 등과 관련한 불확
실성이 크며, 물가의 경우 유가, 환
  4. (p.47.0) 34 
 
2-11. 금년중 소비자물가 상승

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI

llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

In [13]:
prompt = ChatPromptTemplate.from_template("""
당신은 한국은행 경제전망 보고서를 기반으로 답변하는 금융 전문 어시스턴트입니다.
아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.
문서에 없는 내용은 "보고서에서 확인되지 않습니다"라고 답하세요.

[참고문서]
{context}

[질문]
{question}

한글로 간결하고 정확하게 답변하세요.
""")

In [14]:
rag_chain = (
    RunnableParallel(
        context=retriever,
        question=RunnablePassthrough()
    )
    | prompt
    | llm
    | parser
)

In [15]:
# RAG 테스트
test_questions = [
    # 수치 확인용
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "2026년 소비자물가 상승률 전망치는?",
    "수출 전망은 어떻게 되나요?",

    # 범위 밖 질문 (환각 테스트)
    "2030년 경제성장률 전망은?",
    "미국 연방준비제도의 금리 결정 일정은?",

    # 맥락 이해 확인
    "성장률이 2%대로 반등하는 주요 원인은?",
    "미국 관세가 한국 경제에 미치는 영향은?"
]

for q in test_questions:
    print(f"[ Q ] {q}")
    print(f"[ A ] {rag_chain.invoke(q)}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ A ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 2026년 소비자물가 상승률 전망치는?
[ A ] 2026년 소비자물가 상승률 전망치는 2.2%로 예상됩니다.

[ Q ] 수출 전망은 어떻게 되나요?
[ A ] 수출은 견조한 흐름을 이어갈 것으로 전망되며, 2026년에는 통관 기준으로 7,952억 달러에 이를 것으로 예상됩니다.

[ Q ] 2030년 경제성장률 전망은?
[ A ] 보고서에서 확인되지 않습니다.

[ Q ] 미국 연방준비제도의 금리 결정 일정은?
[ A ] 보고서에서 확인되지 않습니다.

[ Q ] 성장률이 2%대로 반등하는 주요 원인은?
[ A ] 보고서에서 확인되지 않습니다.

[ Q ] 미국 관세가 한국 경제에 미치는 영향은?
[ A ] 미국의 관세가 한국 경제에 미치는 영향은 다음과 같습니다. 한국에 대한 미국의 관세율은 현재 13.6%로 유지되고 있으며, 임시관세 15%가 부과될 예정입니다. 이러한 관세 정책은 한국의 수출에 부정적인 영향을 미칠 수 있으며, 특히 반도체와 의약품에 대한 관세 부과가 예정되어 있어 한국의 평균관세율이 상승할 것으로 예상됩니다. 또한, 미국의 관세 정책 변화는 한국을 포함한 전세계 관세에 추가적인 상승 압력을 가할 수 있습니다.



In [16]:
# RAG vs 일반 LLM 비교
llm_base = ChatOpenAI(model="gpt-4o-mini", temperature=0)

compare_questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "미국 관세가 한국 경제에 미치는 영향은?"
]

for q in compare_questions:
    print(f"[ Q ] {q}")
    base_answer = parser.invoke(llm_base.invoke(q))
    print(f"[ 일반 LLM ] {base_answer[:150]}")
    rag_answer = rag_chain.invoke(q)
    print(f"[ RAG 답변 ] {rag_answer[:150]}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ 일반 LLM ] 2026년 GDP 성장률 전망치는 여러 경제 기관과 연구소에 따라 다를 수 있습니다. 일반적으로 이러한 전망치는 경제 상황, 정책 변화, 글로벌 경제 동향 등에 따라 변동할 수 있습니다. 최신 정보를 얻기 위해서는 국제통화기금(IMF), 세계은행, 각국의 중앙은행 또는
[ RAG 답변 ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
[ 일반 LLM ] 소비자물가 상승률 전망은 여러 요인에 따라 달라질 수 있습니다. 일반적으로 경제 성장률, 통화 정책, 원자재 가격, 공급망 문제, 그리고 글로벌 경제 상황 등이 주요한 영향을 미칩니다. 

2023년의 경우, 많은 국가들이 인플레이션 압력을 경험하고 있으며, 이는 에너
[ RAG 답변 ] 소비자물가 상승률은 2026년 중 2.2%로 예상되며, 내년에는 물가 목표 수준인 2%를 나타낼 것으로 보입니다.

[ Q ] 미국 관세가 한국 경제에 미치는 영향은?
[ 일반 LLM ] 미국의 관세가 한국 경제에 미치는 영향은 여러 가지 측면에서 분석할 수 있습니다. 주요 영향은 다음과 같습니다:

1. **수출 감소**: 미국은 한국의 주요 수출 시장 중 하나입니다. 미국이 한국 제품에 높은 관세를 부과하면 한국 기업의 수출 경쟁력이 저하되어 수출량
[ RAG 답변 ] 미국의 관세가 한국 경제에 미치는 영향은 다음과 같습니다. 한국에 대한 미국의 관세율은 현재 13.6%로 유지되고 있으며, 임시관세 15%가 부과될 예정입니다. 이러한 관세 정책은 한국의 수출에 부정적인 영향을 미칠 수 있으며, 특히 반도체와 의약품에 대한 관세 부과가

